<a href="https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
display(df.head())

Shape: (9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Rule

Prioritize content pages that have meaningful search visibility but a relatively weak average search position. The goal is to identify pages that may have a ranking opportunity and are worth reviewing.

# Reason code

- `high_visibility_low_position` — the page has relatively high impressions and a relatively weak average position.

# Action

- `review_ranking_opportunity` — review the page and its search intent before recommending an optimization.

In [11]:
import numpy as np

# Create buckets for the two signals
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, 50, float("inf")],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 10, 100, 1000, 10000, float("inf")],
    labels=["0-10", "11-100", "101-1000", "1001-10000", "10000+"]
)

print("=== Signal 1: Average Position ===")

position_check = (
    df["position_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

display(position_check)

print("=== Signal 2: Impressions ===")

impression_check = (
    df["impression_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("impression_bucket")
    .reset_index(name="n")
)

display(impression_check)

=== Signal 1: Average Position ===


,position_bucket,n
0,1-3,727362
1,4-5,535763
2,6-10,920359
3,11-20,519223
4,21-50,631491
5,50+,276863
6,NaN,6230317


=== Signal 2: Impressions ===


,impression_bucket,n
0,0-10,7761951
1,11-100,1445944
2,101-1000,601123
3,1001-10000,32232
4,10000+,128


In [12]:
print("=== Position signal summary ===")

position_summary = (
    df.groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

display(position_summary)

print("=== Impression signal summary ===")

impression_summary = (
    df.groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        avg_position=("gsc_avg_position", "mean"),
        avg_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

display(impression_summary)

=== Position signal summary ===


,position_bucket,n,avg_impressions,avg_clicks
0,1-3,727362,74.280199,0.282469
1,4-5,535763,126.686331,0.449843
2,6-10,920359,76.009757,0.222543
3,11-20,519223,56.596118,0.178053
4,21-50,631491,88.590989,0.121283
5,50+,276863,12.527727,0.005450


=== Impression signal summary ===


,impression_bucket,n,avg_position,avg_clicks
0,0-10,7761951,20.399400,0.002120
1,11-100,1445944,13.068357,0.108940
2,101-1000,601123,11.022364,0.804609
3,1001-10000,32232,11.919689,4.837491
4,10000+,128,3.784447,64.593750


### Signal verdicts

- `gsc_avg_position` — **CONFIRMED**: The buckets show a directional relationship between position and search performance, so position is useful as a prioritization signal.
- `gsc_impressions` — **MIXED**: Higher visibility identifies pages with more search exposure, but impressions alone do not establish that a page is a good optimization opportunity.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# Rank signals as percentiles

df["impressions_rank"] = df["gsc_impressions"].rank(
    pct=True,
    method="average"
)

df["position_rank"] = df["gsc_avg_position"].rank(
    pct=True,
    method="average"
)

# Higher score = higher review priority
df["action_score"] = (
    0.5 * df["impressions_rank"]
    + 0.5 * df["position_rank"]
)

# One reason code and one action
df["reason_code"] = "high_visibility_low_position"
df["action"] = "review_ranking_opportunity"

# Build ranked queue
queue = df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_avg_position",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

queue = queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

print("Queue size:", len(queue))

display(queue.head(20))

Queue size: 9841378


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,action_score,reason_code,action
0,1,client_20259bd6705d81d4,content_1c14838b50c55c9f,2026-03-25,1190,134.111765,0.998694,high_visibility_low_position,review_ranking_opportunity
1,2,client_20259bd6705d81d4,content_f4bd56290b265782,2026-03-25,1146,105.913613,0.998492,high_visibility_low_position,review_ranking_opportunity
2,3,client_20259bd6705d81d4,content_b6259d89f57d5ebc,2026-03-25,986,96.248479,0.996996,high_visibility_low_position,review_ranking_opportunity
3,4,client_20259bd6705d81d4,content_6c67f84d8b90e7c5,2026-03-25,689,118.474601,0.996719,high_visibility_low_position,review_ranking_opportunity
4,5,client_20259bd6705d81d4,content_dbdef2d26fa09819,2026-03-25,667,117.287856,0.996541,high_visibility_low_position,review_ranking_opportunity
5,6,client_20259bd6705d81d4,content_4c8f60c8875f8cc6,2026-03-25,640,99.345313,0.996198,high_visibility_low_position,review_ranking_opportunity
6,7,client_20259bd6705d81d4,content_3896bdc4b2b82656,2026-03-25,617,121.458671,0.996122,high_visibility_low_position,review_ranking_opportunity
7,8,client_20259bd6705d81d4,content_4c82c93a3ee8cba8,2026-03-25,560,149.058929,0.995586,high_visibility_low_position,review_ranking_opportunity
8,9,client_23a62021009f63c4,content_6530fa9d297c46eb,2026-03-31,5364,89.848248,0.995552,high_visibility_low_position,review_ranking_opportunity
9,10,client_20259bd6705d81d4,content_77d3a2a3d81fd40c,2026-03-25,571,101.861646,0.995544,high_visibility_low_position,review_ranking_opportunity


In [15]:
import os

output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(queue))

Saved: /content/work/outputs/baseline_action_score.csv
Rows: 9841378


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
top20 = queue.head(20).copy()

display(top20)

,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,action_score,reason_code,action
0,1,client_20259bd6705d81d4,content_1c14838b50c55c9f,2026-03-25,1190,134.111765,0.998694,high_visibility_low_position,review_ranking_opportunity
1,2,client_20259bd6705d81d4,content_f4bd56290b265782,2026-03-25,1146,105.913613,0.998492,high_visibility_low_position,review_ranking_opportunity
2,3,client_20259bd6705d81d4,content_b6259d89f57d5ebc,2026-03-25,986,96.248479,0.996996,high_visibility_low_position,review_ranking_opportunity
3,4,client_20259bd6705d81d4,content_6c67f84d8b90e7c5,2026-03-25,689,118.474601,0.996719,high_visibility_low_position,review_ranking_opportunity
4,5,client_20259bd6705d81d4,content_dbdef2d26fa09819,2026-03-25,667,117.287856,0.996541,high_visibility_low_position,review_ranking_opportunity
5,6,client_20259bd6705d81d4,content_4c8f60c8875f8cc6,2026-03-25,640,99.345313,0.996198,high_visibility_low_position,review_ranking_opportunity
6,7,client_20259bd6705d81d4,content_3896bdc4b2b82656,2026-03-25,617,121.458671,0.996122,high_visibility_low_position,review_ranking_opportunity
7,8,client_20259bd6705d81d4,content_4c82c93a3ee8cba8,2026-03-25,560,149.058929,0.995586,high_visibility_low_position,review_ranking_opportunity
8,9,client_23a62021009f63c4,content_6530fa9d297c46eb,2026-03-31,5364,89.848248,0.995552,high_visibility_low_position,review_ranking_opportunity
9,10,client_20259bd6705d81d4,content_77d3a2a3d81fd40c,2026-03-25,571,101.861646,0.995544,high_visibility_low_position,review_ranking_opportunity


### Top-20 review

For each selected page, the action is `review_ranking_opportunity` and the reason code is `high_visibility_low_position`.

The selections are prioritization candidates, not automatic recommendations. A page could be a weak pick if its impressions come from low-value queries, if the average position is unstable, if the page is already improving, or if the underlying data is incomplete.

In [18]:
top20_review = queue.head(20).copy()

top20_review["confidence_note"] = (
    "Baseline priority based on impressions and average position; "
    "requires manual review."
)

top20_review["what_would_make_it_wrong"] = (
    "Low-value queries, unstable position, incomplete data, "
    "or an already-improving page."
)

top20_review = top20_review[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_1c14838b50c55c9f,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
1,2,content_f4bd56290b265782,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
2,3,content_b6259d89f57d5ebc,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
3,4,content_6c67f84d8b90e7c5,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
4,5,content_dbdef2d26fa09819,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
5,6,content_4c8f60c8875f8cc6,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
6,7,content_3896bdc4b2b82656,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
7,8,content_4c82c93a3ee8cba8,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
8,9,content_6530fa9d297c46eb,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."
9,10,content_77d3a2a3d81fd40c,review_ranking_opportunity,high_visibility_low_position,Baseline priority based on impressions and ave...,"Low-value queries, unstable position, incomple..."


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
print("=== Weak picks ===")

weak_picks = queue.tail(10)[
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "action_score",
        "reason_code",
        "action"
    ]
]

display(weak_picks)

=== Weak picks ===


,rank,content_hash_id,gsc_impressions,gsc_avg_position,action_score,reason_code,action
9841368,9841369,content_cace6935c8da0913,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841369,9841370,content_f85063e1a795eb54,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841370,9841371,content_250770b04a9318e7,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841371,9841372,content_30b2a310a91cc470,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841372,9841373,content_d9c23945d9fa7311,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841373,9841374,content_dfc972c4709a0b42,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841374,9841375,content_a5b7be083fcba9a4,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841375,9841376,content_0b88e713bfa1eae9,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841376,9841377,content_dd1deec61eb35197,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity
9841377,9841378,content_d105e79bf855829c,0,NaN,NaN,high_visibility_low_position,review_ranking_opportunity


### Weak picks

Some selections can still be wrong because the baseline only uses two signals. High impressions do not necessarily mean that a page is a valuable optimization opportunity, and a weak average position can be caused by competitive queries or search intent mismatch. Therefore, the action is a review recommendation rather than an automatic content change.

### Leakage check

The baseline uses only March 2026 observations and does not use future months, future outcomes, product flags, trend labels, or label-derived fields. The score is based only on `gsc_impressions` and `gsc_avg_position`, which are available in the decision window.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.